# 07 — Model comparison

Loads the pipeline outputs and produces the comparison tables and figures used
in the report. Run `python scripts/run_pipeline.py` first.

**Report sections fed:** 9 (Results and error analysis), 11 (Conclusion).


In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from appliance_energy import config, data, evaluation, features, plotting, stationarity
from appliance_energy.models import benchmarks, feature_models, foundation, sarimax

pd.set_option("display.width", 140)


In [ ]:
frame = data.load_hourly()
y = frame[config.TARGET]

y_train, y_test = data.train_test_split(y)
test_index = y_test.index

print(f"train {y_train.index.min()} -> {y_train.index.max()}  ({len(y_train)})")
print(f"test  {test_index.min()} -> {test_index.max()}  ({len(y_test)})")


In [ ]:
forecast_df = pd.read_csv(config.FORECAST_DIR / "all_forecasts.csv",
                          index_col=0, parse_dates=True)
results = pd.read_csv(config.METRICS_DIR / "model_comparison.csv")
results.round(3)


### Report table\n\nPaste into report Section 9.1.

In [ ]:
print(results.round(3).to_markdown(index=False))


## Figures

In [ ]:
fig = plotting.plot_forecast_window(forecast_df, window_hours=72)
fig.savefig("../reports/figures/forecast_comparison.png", dpi=200, bbox_inches="tight")


In [ ]:
fig = plotting.plot_forecast_panel(forecast_df)
fig.savefig("../reports/figures/forecast_panel.png", dpi=200, bbox_inches="tight")


## Error by lead time\n\nThe strongest evidence for what each model has learned.

In [ ]:
forecasts = {c: forecast_df[c] for c in forecast_df.columns if c != "actual"}
lead = evaluation.errors_by_horizon(forecasts, forecast_df["actual"], config.HORIZON)

fig = plotting.plot_error_by_lead_time(lead)
fig.savefig("../reports/figures/error_by_lead_time.png", dpi=200, bbox_inches="tight")
lead.round(2)


## Diebold–Mariano tests

Which differences in Section 9.1 are statistically meaningful, and which fall
within what a different test fortnight might reverse? Loss differentials are
serially correlated, so a HAC variance estimator is required.


In [ ]:
from scipy import stats

def diebold_mariano(actual, f1, f2, h=24):
    d = (actual - f1).abs() - (actual - f2).abs()
    d = d.dropna()
    n = len(d)

    # Newey-West variance with h-1 lags
    gamma0 = d.var(ddof=0)
    var = gamma0
    for lag in range(1, h):
        cov = np.cov(d[lag:], d[:-lag])[0, 1]
        var += 2 * (1 - lag / h) * cov

    dm = d.mean() / np.sqrt(var / n)
    return dm, 2 * (1 - stats.norm.cdf(abs(dm)))

best = results["model"].iloc[0]
for other in results["model"].iloc[1:]:
    dm, p = diebold_mariano(forecast_df["actual"], forecast_df[best], forecast_df[other])
    flag = "significant" if p < 0.05 else "not significant"
    print(f"{best} vs {other:<24} DM {dm:+.2f}  p {p:.3f}  {flag}")


## Residual diagnostics for the leading model

In [ ]:
residuals = forecast_df[best] - forecast_df["actual"]
fig = plotting.plot_residual_diagnostics(residuals)
fig.savefig("../reports/figures/residual_diagnostics.png", dpi=200, bbox_inches="tight")


### Bias relative to error magnitude\n\nBias approaching half of MAE indicates a systematic rather than noisy error.

In [ ]:
summary = results.set_index("model")[["MAE", "Bias"]].copy()
summary["|Bias| / MAE"] = (summary["Bias"].abs() / summary["MAE"]).round(3)
summary.round(3)
